In [ ]:
!pip install gradio folium gtts

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 3.4 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1


In [ ]:
# 📦 Imports
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import numpy as np
import folium
import gradio as gr
from gtts import gTTS
import os
from statsmodels.tsa.arima.model import ARIMA
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [ ]:
# ✅ Load AQI dataset
FILE_PATH = '/content/cleaned_air_quality.xlsx'

try:
    df = pd.read_excel(FILE_PATH)
    print("✅ Dataset loaded successfully")

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    REQUIRED_COLUMNS = {"so2", "no2", "rspm", "spm", "location", "date"}
    missing_columns = REQUIRED_COLUMNS - set(df.columns)

    if missing_columns:
        print(f"⚠ Missing columns: {missing_columns}")
        df = None
    else:
        df = df.dropna(subset=REQUIRED_COLUMNS)
except Exception as e:
    print(f"❌ Error loading dataset: {e}")
    df = None

if df is not None:
    df = df[df['date'].dt.year >= 2011]
    df.drop(columns=['agency', 'location_monitoring_station', 'type'], inplace=True, errors='ignore')
    for col in ['so2', 'no2', 'rspm', 'spm']:
        df[col].fillna(df[col].mean(), inplace=True)
    for col in ['state', 'location']:
        df[col].fillna(df[col].mode()[0], inplace=True)

✅ Dataset loaded successfully


/tmp/ipython-input-753523780.py:25: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mean(), inplace=True)
/tmp/ipython-input-753523780.py:27: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 

In [ ]:
# ✅ Add AQI and category
def categorize_aqi(aqi):
    if aqi <= 50:
        return 'Good'
    elif aqi <= 100:
        return 'Satisfactory'
    elif aqi <= 200:
        return 'Moderate'
    elif aqi <= 300:
        return 'Poor'
    elif aqi <= 400:
        return 'Very Poor'
    else:
        return 'Severe'

df['aqi'] = df[['so2', 'no2', 'rspm', 'spm']].mean(axis=1)
df['aqi_category'] = df['aqi'].apply(categorize_aqi)

le_category = LabelEncoder()
df['category_enc'] = le_category.fit_transform(df['aqi_category'])

features = ['so2', 'no2', 'rspm', 'spm']
X = df[features]
y_reg = df['aqi']
y_cls = df['category_enc']

X_train, X_test, y_reg_train, y_reg_test, y_cls_train, y_cls_test = train_test_split(
    X, y_reg, y_cls, test_size=0.2, random_state=42
)

regressor = RandomForestRegressor(n_estimators=100, random_state=42)
regressor.fit(X_train, y_reg_train)

classifier = RandomForestClassifier(n_estimators=100, random_state=42)
classifier.fit(X_train, y_cls_train)

RandomForestClassifier(random_state=42)

In [ ]:
# ✅ Prediction function using ML
def ml_predict_aqi(so2, no2, rspm, spm):
    input_df = pd.DataFrame([[so2, no2, rspm, spm]], columns=features)
    aqi_pred = regressor.predict(input_df)[0]
    category_pred_enc = classifier.predict(input_df)[0]
    category = le_category.inverse_transform([category_pred_enc])[0]
    return f"{aqi_pred:.2f}", category

In [ ]:
# ✅ Existing AQI calculation (used for city tab)
def get_aqi_condition(aqi):
    if aqi <= 50:
        return "अच्छा", "Good"
    elif aqi <= 100:
        return "संतोषजनक", "Satisfactory"
    elif aqi <= 200:
        return "मध्यम", "Moderate"
    elif aqi <= 300:
        return "खराब", "Poor"
    elif aqi <= 400:
        return "बहुत खराब", "Very Poor"
    else:
        return "गंभीर", "Severe"


def city_voice_report(city_name, lang):
    city_df = df[df['location'] == city_name].copy()
    if city_df.empty:
        msg = "Sorry, no data available for this city."
        tts = gTTS(text=msg, lang='en')
        tts.save("voice.mp3")
        return msg, "voice.mp3"

    aqi_val = city_df[['so2', 'no2', 'rspm', 'spm']].mean(axis=1).mean()
    hindi_cond, eng_cond = get_aqi_condition(aqi_val)

    if lang == "English":
        msg = f"The current AQI in {city_name} is {aqi_val:.2f}, which is considered {eng_cond}."
        lang_code = 'en'
    elif lang == "Hindi":
        msg = f"{city_name} का AQI {aqi_val:.2f} है, जो '{hindi_cond}' श्रेणी में आता है।"
        lang_code = 'hi'
    elif lang == "Kannada":
        msg = f"{city_name} ನ AQI {aqi_val:.2f} ಆಗಿದ್ದು, ಇದು '{eng_cond}' ವರ್ಗವಾಗಿ ಪರಿಗಣಿಸಲಾಗಿದೆ."
        lang_code = 'kn'
    elif lang == "Tamil":
        msg = f"{city_name}-இல் தற்போதைய காற்றின் தரம் {aqi_val:.2f}, இது '{eng_cond}' என வகைப்படுத்தப்படுகிறது."
        lang_code = 'ta'
    elif lang == "Telugu":
        msg = f"{city_name} లో AQI {aqi_val:.2f}, ఇది '{eng_cond}' గా పరిగణించబడుతుంది."
        lang_code = 'te'
    else:
        msg = f"AQI for {city_name} is {aqi_val:.2f}."
        lang_code = 'en'

    tts = gTTS(text=msg, lang=lang_code)
    tts.save("voice.mp3")
    return msg, "voice.mp3"

In [ ]:
def calculate_aqi(so2, no2, rspm, spm):
    return round(np.mean([so2, no2, rspm, spm]), 2), categorize_aqi(np.mean([so2, no2, rspm, spm]))

def city_avg_aqi(city_name):
    city_df = df[df['location'] == city_name].copy()
    if city_df.empty:
        return f"No data for {city_name}", None

    city_df["aqi"] = city_df[['so2', 'no2', 'rspm', 'spm']].mean(axis=1)

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.lineplot(data=city_df, x="date", y="aqi", label="Predicted AQI", ax=ax)
    ax.set_title(f"Predicted AQI Trend in {city_name}")
    ax.set_xlabel("Date")
    ax.set_ylabel("Predicted AQI")
    ax.legend()
    ax.grid(True)

    avg_aqi = city_df["aqi"].mean()
    level = categorize_aqi(avg_aqi)

    return f"AQI: {avg_aqi:.2f} - {level}", fig


In [ ]:
def update_scatter(selected_location):
    if df is None or selected_location is None:
        return px.scatter(title="No Data Available")
    filtered_df = df[df["location"] == selected_location]
    return px.scatter(filtered_df, x="date", y="so2", title="SO2 Levels Over Time", template="plotly_white")

def update_time_series(selected_location):
    if df is None or selected_location is None:
        return px.line(title="No Data Available")
    filtered_df = df[df["location"] == selected_location]
    return px.line(filtered_df, x="date", y=["no2", "rspm"], title="NO2 & RSPM Levels Over Time", template="plotly_white")

def update_bar_chart(selected_location):
    if df is None or selected_location is None:
        return px.bar(title="No Data Available")
    filtered_df = df[df["location"] == selected_location]
    return px.bar(filtered_df, x="date", y="spm", title="SPM Levels Over Time", template="plotly_dark", color_discrete_sequence=["#FFFFFF"])

In [ ]:
def get_precautions(avg_aqi):
        level = categorize_aqi(avg_aqi)
        tips = {
            'Good': 'Enjoy your outdoor activities! Air quality is safe.',
            'Satisfactory': 'Slight discomfort to sensitive individuals. Enjoy but stay aware.',
            'Moderate': 'Avoid prolonged outdoor exertion if sensitive. Stay hydrated.',
            'Poor': 'Limit outdoor activities. Use masks. Stay indoors if possible.',
            'Very Poor': 'Avoid outdoor exposure. Use air purifiers. Mask up!',
            'Severe': 'Serious health effects. Stay indoors. Avoid all physical exertion.'
        }
        return f"AQI Level: {level}\n\n{tips[level]}"

def health_advice(avg_aqi, health_profile):
    level = categorize_aqi(avg_aqi)
    tips = {
        'Good': {
            'Asthma': 'No issues expected. Enjoy your day!',
            'COPD': 'No restrictions. Keep medications handy.',
            'None': 'All clear!'
        },
        'Satisfactory': {
            'Asthma': 'Minor symptoms possible. Carry inhaler.',
            'COPD': 'Mild discomfort. Prefer indoor activities.',
            'None': 'Stay aware, especially if sensitive.'
        },
        'Moderate': {
            'Asthma': 'Avoid intense outdoor activities. Use inhaler.',
            'COPD': 'Use mask. Reduce outdoor exposure.',
            'None': 'Limit heavy exercise outdoors.'
        },
        'Poor': {
            'Asthma': 'Stay indoors. Severe symptoms likely outdoors.',
            'COPD': 'Indoor stay recommended. Use air purifier.',
            'None': 'Use masks, avoid polluted areas.'
        },
        'Very Poor': {
            'Asthma': 'Emergency-level pollution. Stay indoors with purifier.',
            'COPD': 'Consult doctor if needed. Avoid going out.',
            'None': 'Health impacts for everyone. Use protection.'
        },
        'Severe': {
            'Asthma': 'Critical danger. Emergency care if symptoms worsen.',
            'COPD': 'Very hazardous. Stay indoors completely.',
            'None': 'Stay inside. Avoid any exertion or exposure.'
        }
    }
    return f"AQI Level: {level}\n\n{tips[level].get(health_profile, tips[level]['None'])}"


def policy_suggestions(city_name, avg_aqi):
    level = categorize_aqi(avg_aqi)
    suggestions = []

    if avg_aqi > 400:
        suggestions = [
            "❌ Ban construction & industrial activity.",
            "🚫 Implement complete vehicle ban in hotspots.",
            "🚨 Public emergency announcements daily."
        ]
    elif avg_aqi > 300:
        suggestions = [
            "🔁 Implement odd-even vehicle rules.",
            "🏭 Restrict factory emissions & dust control.",
            "🌳 Plan immediate urban afforestation drives."
        ]
    elif avg_aqi > 200:
        suggestions = [
            "👥 Encourage remote work in schools/offices.",
            "🚌 Promote public transport, increase buses.",
            "🚷 Limit outdoor events and gatherings."
        ]
    elif avg_aqi > 100:
        suggestions = [
            "💬 Alert citizens on mobile apps.",
            "🧹 Conduct road cleaning and anti-dust measures."
        ]
    else:
        suggestions = ["✅ Air quality is acceptable. Continue regular monitoring."]

    return f"Policy Suggestions for {city_name} (AQI: {avg_aqi}, Level: {level}):\n\n" + "\n".join(suggestions)


In [ ]:
# ✅ ARIMA Forecasting
years = np.arange(2000, 2021)
so2_levels = np.random.randint(10, 50, size=len(years))
no2_levels = np.random.randint(20, 80, size=len(years))
rspm_levels = np.random.randint(30, 100, size=len(years))
spm_levels = np.random.randint(40, 120, size=len(years))

df_yearly = pd.DataFrame({
    "year": years,
    "so2": so2_levels,
    "no2": no2_levels,
    "rspm": rspm_levels,
    "spm": spm_levels
})

def arima_forecast(pollutant, steps):
    pollutant_series = df_yearly.set_index("year")[pollutant]
    model = ARIMA(pollutant_series, order=(1, 1, 1))
    model_fit = model.fit()
    forecast = model_fit.forecast(steps=steps)
    forecast_years = np.arange(df_yearly["year"].max() + 1, df_yearly["year"].max() + 1 + steps)

    plt.figure(figsize=(10, 5))
    plt.plot(pollutant_series, label=f"Actual {pollutant.upper()}", marker='o')
    plt.plot(forecast_years, forecast, label=f"ARIMA Prediction for {pollutant.upper()}", linestyle="dashed", color="red", marker='o')
    plt.xlabel("Year")
    plt.ylabel(f"{pollutant.upper()} Levels")
    plt.legend()
    plt.grid()
    plt.title(f"{pollutant.upper()} Level Forecast using ARIMA (1,1,1)")
    return plt.gcf()

In [ ]:
# ✅ Map
geo_df = pd.read_excel("/content/reshaped_data.xlsx", sheet_name="Geolocation")

state_data = {}
for _, row in geo_df.iterrows():
    state = row["State"]
    city = row["City"]
    lat = row["Latitude"]
    lon = row["Longitude"]
    aqi = row["aqi"]

    if state not in state_data:
        state_data[state] = {"center": (lat, lon), "cities": {}}

    state_data[state]["cities"][city] = {
        "coords": (lat, lon),
        "aqi": aqi
    }


def update_cities(state):
    if state in state_data:
        return gr.update(choices=["(Show Entire State)"] + list(state_data[state]["cities"].keys()), value="(Show Entire State)")
    return gr.update(choices=[], value=None)

def generate_map(state, city):
    if state in state_data:
        if city == "(Show Entire State)":
            lat, lon = state_data[state]["center"]
            zoom_level = 7
            markers = state_data[state]["cities"]
        else:
            lat, lon = state_data[state]["cities"][city]["coords"]
            zoom_level = 10
            markers = {city: state_data[state]["cities"][city]}

        city_map = folium.Map(location=[lat, lon], zoom_start=zoom_level, tiles="CartoDB positron")
        for city_name, data in markers.items():
            coords = data["coords"]
            aqi = data.get("aqi", None)
            color = get_color_for_aqi(aqi) if aqi is not None else "gray"
            popup_text = f"{city_name}, {state}<br>AQI: {aqi:.2f}" if aqi is not None else f"{city_name}, {state}<br>AQI: N/A"
            folium.Marker(
                coords,
                popup=popup_text,
                tooltip=city_name,
                icon=folium.Icon(color=color)
            ).add_to(city_map)
        return city_map._repr_html_()
    return "<h3>No map available</h3>"


# ✅ State AQI Color Map Logic
def get_color_for_aqi(aqi):
    if aqi <= 50:
        return 'green'
    elif aqi <= 100:
        return 'darkgreen'
    elif aqi <= 200:
        return 'orange'
    elif aqi <= 300:
        return 'red'
    elif aqi <= 400:
        return 'purple'
    else:
        return 'maroon'

def generate_state_colored_map(selected_state):
    if df is None or selected_state not in state_data:
        return "<h3>No data available or invalid state selected</h3>"

    state_avg = df.groupby('state')['aqi'].mean().to_dict()
    india_center = [22.9734, 78.6569]

    # Create the base map centered on India
    map_obj = folium.Map(location=india_center, zoom_start=5, tiles="CartoDB positron")

    # Get AQI for the selected state
    avg_aqi = state_avg.get(selected_state)
    if avg_aqi is None:
        return f"<h3>No AQI data for {selected_state}</h3>"

    state_info = state_data[selected_state]
    color = get_color_for_aqi(avg_aqi)
    popup_text = f"{selected_state}: {avg_aqi:.2f} AQI ({categorize_aqi(avg_aqi)})"

    # Add a full-size colored Circle representing the state
    folium.Circle(
        location=state_info["center"],
        radius=120000,  # You can adjust the radius for visual scale
        popup=popup_text,
        tooltip=popup_text,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.6
    ).add_to(map_obj)

    return map_obj._repr_html_()

# Pollutant-wise precautions
pollutant_measures = {
    "PM2.5": "Use air purifiers indoors, wear N95 masks, avoid outdoor activity.",
    "PM10": "Avoid dusty areas, wet mop floors, stay indoors during dust storms.",
    "NO2": "Reduce car use, maintain vehicles, avoid fossil fuel indoors.",
    "SO2": "Stay indoors during industrial emissions, use air filters.",
    "CO": "Ensure ventilation, avoid closed spaces with engines, install CO detectors.",
    "O3": "Avoid sun exposure in afternoons, use public transport, limit sprays."
}

# AQI level precautions
aqi_level_measures = {
    "Good (0-50)": "✅ Satisfactory. No precautions needed.",
    "Satisfactory (51-100)": "🙂 Sensitive people should limit outdoor exposure.",
    "Moderate (101-200)": "⚠️ Sensitive groups avoid exertion outdoors.",
    "Poor (201-300)": "❗ Avoid outdoor activity. Wear protective masks.",
    "Very Poor (301-400)": "🚫 Stay indoors, use purifiers, avoid all physical effort.",
    "Severe (401-500)": "🆘 Emergency! Avoid all outdoor exposure completely."
}

# SDG info
sdg_data = {
    "SDG Goal": [
        "SDG 3: Good Health & Wellbeing",
        "SDG 11: Sustainable Cities",
        "SDG 13: Climate Action",
        "SDG 9: Industry, Innovation"
    ],
    "Contribution": [
        "Reduces exposure via early alerts",
        "Promotes urban air quality awareness",
        "Encourages pollution monitoring",
        "Applies AI/ML for sustainability"
    ]
}
sdg_df = pd.DataFrame(sdg_data)

def extract_aqi(city_info_text):
    try:
        # Example format: "City: Delhi | AQI: 134"
        aqi = int(city_info_text.split("AQI:")[1].strip())
        return aqi
    except:
        return None


In [ ]:
location_list = df['location'].unique().tolist() if df is not None else []

In [ ]:
location_list = df['location'].unique().tolist() if df is not None else []

with gr.Blocks(title="AirAware - Smart AQI Monitor") as demo:
    with gr.Tab("📘 About & Info"):
        gr.Markdown("""<div style="text-align: center; margin-bottom: 50px;">
            <h1 style="font-size: 3em; color: #2C3E50;">🌿 AirAware</h1>
            <h3 style="color: #7F8C8D;">Breathe Smart. Live Aware. Monitor AQI in Real-Time</h3>
            <hr style="width:50%; margin:auto;">
        </div>""")

        gr.Markdown("""## 📘 About This Platform

        *AirAware* is a machine learning-powered air quality monitoring system aimed at creating awareness
        and providing precautionary measures against pollutants. It helps users take proactive steps towards
        cleaner air and a healthier lifestyle.

        - *Pollutants Tracked:* PM2.5, PM10, NO₂, SO₂, CO, and O₃
        - *Built With:* Python, ML Models, Gradio Interface
        - *Use Case:* Government planning, urban awareness, health safety
        """)

        gr.Markdown("## 🔬 Pollutant-wise Precautions")
        gr.Dataframe(pd.DataFrame({
            "Pollutant": list(pollutant_measures.keys()),
            "Precaution Tips": list(pollutant_measures.values())
        }), interactive=False)

        gr.Markdown("## 📊 AQI Levels & Health Recommendations")
        gr.Dataframe(pd.DataFrame({
            "AQI Level": list(aqi_level_measures.keys()),
            "Recommended Action": list(aqi_level_measures.values())
        }), interactive=False)

        gr.Markdown("## 🌍 UN Sustainable Development Goals (SDGs) Aligned")
        gr.Dataframe(sdg_df, interactive=False)

        gr.Markdown("""## 📩 Contact Us

        Have suggestions, queries, or collaborations? Reach out!

        - 📧 Email: [airaware@domain.com](mailto:airaware@domain.com)
        - 🌐 Website: [www.airaware.com](https://www.airaware.com)
        - 💬 LinkedIn: [linkedin.com/in/airaware](https://www.linkedin.com/in/airaware)""")

        gr.Markdown("""<div style="text-align: center; font-size: 0.9em; color: grey;">
            © 2025 AirAware • Built at Amrita School of Computing, Bengaluru 🌱
        </div>""")

    with gr.Tab("🔮 Predict AQI"):
        so2_input = gr.Number(value=df['so2'].mean(), label="SO2 Level")
        no2_input = gr.Number(value=df['no2'].mean(), label="NO2 Level")
        rspm_input = gr.Number(value=df['rspm'].mean(), label="RSPM Level")
        spm_input = gr.Number(value=df['spm'].mean(), label="SPM Level")
        predict_btn = gr.Button("Predict AQI")
        pred_output = gr.Textbox(label="Predicted AQI")
        category_output = gr.Textbox(label="AQI Category")
        predict_btn.click(fn=ml_predict_aqi, inputs=[so2_input, no2_input, rspm_input, spm_input], outputs=[pred_output, category_output])

    with gr.Tab("📊 City AQI Overview"):
      city_input = gr.Dropdown(choices=location_list, label="Select City")
      city_btn = gr.Button("Get City AQI & Trends")
      city_output = gr.Textbox(label="City AQI Info")
      plot_output = gr.Plot(label="Predicted AQI Trend")

      lang_dropdown = gr.Dropdown(["English", "Hindi", "Kannada", "Tamil", "Telugu"], label="Select Language for Voice Report")
      voice_btn = gr.Button("🔊 Generate AQI Voice Report")
      voice_text = gr.Textbox(label="Voice Message")
      voice_audio = gr.Audio(label="Voice Output", autoplay=True)

      city_btn.click(city_avg_aqi, inputs=[city_input], outputs=[city_output, plot_output])
      voice_btn.click(city_voice_report, inputs=[city_input, lang_dropdown], outputs=[voice_text, voice_audio])
      """
      # 🫁 Health Advice - no manual AQI input
      gr.Markdown("### 🫁 Health Advice")
      health_input = gr.Dropdown(["None", "Asthma", "COPD"], label="Select Health Condition")
      health_btn = gr.Button("Get Health Advice")
      health_output = gr.Text()

      # Automatically extract AQI from city_output and pass it
      health_btn.click(fn=lambda text, cond: health_advice(extract_aqi(text), cond),
                      inputs=[city_output, health_input],
                      outputs=health_output)

      # 📋 General AQI Safety Tips - use AQI from city_output
      gr.Markdown("### 📋 General AQI Safety Tips")
      tip_output = gr.Text()
      city_output.change(fn=lambda text: get_precautions(extract_aqi(text)),
                        inputs=city_output,
                        outputs=tip_output)

      # 🏛 Policy Suggestions
      gr.Markdown("### 🏛 Policy Suggestions")
      policy_city = gr.Textbox(label="City Name")
      policy_aqi = gr.Number(label="AQI Value")
      policy_btn = gr.Button("Suggest Policies")
      policy_output = gr.Text()
      policy_btn.click(policy_suggestions, inputs=[policy_city, policy_aqi], outputs=policy_output)
      """


    with gr.Tab("📈 Interactive Plotly Dashboard"):
        location_dropdown = gr.Dropdown(choices=location_list, label="🔍 Select a Location", interactive=True)
        scatter_plot = gr.Plot(label="SO2 Levels Over Time")
        time_series_plot = gr.Plot(label="NO2 & RSPM Levels Over Time")
        bar_chart = gr.Plot(label="SPM Levels Over Time")
        location_dropdown.change(update_scatter, inputs=location_dropdown, outputs=scatter_plot)
        location_dropdown.change(update_time_series, inputs=location_dropdown, outputs=time_series_plot)
        location_dropdown.change(update_bar_chart, inputs=location_dropdown, outputs=bar_chart)

    with gr.Tab("📈 ARIMA Forecasting"):
        pollutant_input = gr.Dropdown(["so2", "no2", "rspm", "spm"], label="Select Pollutant")
        year_slider = gr.Slider(1, 10, step=1, value=5, label="Forecast Years")
        forecast_plot = gr.Plot()
        forecast_btn = gr.Button("Forecast")
        forecast_btn.click(fn=arima_forecast, inputs=[pollutant_input, year_slider], outputs=forecast_plot)

    with gr.Tab("🗺️ City Map"):
        gr.Markdown("### Interactive State & City Map (with Geolocation)")
        state_dropdown = gr.Dropdown(label="Select State", choices=list(state_data.keys()), interactive=True)
        city_dropdown = gr.Dropdown(label="Select City", choices=[], interactive=True)
        map_display = gr.HTML(value="<h3>Select a state or city to see the map</h3>")
        state_dropdown.change(update_cities, inputs=state_dropdown, outputs=city_dropdown)
        city_dropdown.change(generate_map, inputs=[state_dropdown, city_dropdown], outputs=map_display)

    with gr.Tab("🧭 State AQI Color Map"):
        gr.Markdown("### State-Wise AQI Map with Color Indicators")
        selected_state = gr.Dropdown(label="Select State", choices=list(state_data.keys()))
        state_map_output = gr.HTML(value="<h3>Select a state to view its AQI color</h3>")
        selected_state.change(fn=generate_state_colored_map, inputs=selected_state, outputs=state_map_output)

# ---------------- Launch the Interface ----------------
if __name__ == "__main__":
    demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fb8205a59857291b89.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning:

An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning:

An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning:

An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning:

No supported index is available. Prediction results will be given with an integer index beginning at `start`.

/usr/local/lib/python3.12/dist-packages/st